# Serve the adapter with vLLM on Colab: hot-swap A/B in one run

**Before running**: Runtime -> Change runtime type -> **T4 or L4 GPU**.
You also need `adapters.zip` (downloaded from `colab_t4_run.ipynb`) on your
local machine.

This demonstrates the serving half of the lab: one vLLM server loads the
fp16 base **once** and exposes the LoRA adapter as a second model name
(`ticket`). The client picks base or adapter per request by model name -
that is adapter hot-swapping, the '1 GPU + N small files serves N variants'
story. QLoRA trains on a 4-bit base but serves on the fp16 base; that is
standard practice and the quality delta is negligible.

In [ ]:
# 1. Check what CUDA your driver supports BEFORE installing vllm.
#    Look at 'CUDA Version: X.Y' in the top-right of the output.
!nvidia-smi | head -4

### Install vLLM - match the wheel to your CUDA
The #1 self-hosting pitfall is a wheel built for a different CUDA than the
environment (symptom: `ImportError: libcudart.so.13: cannot open shared
object file`). Pick ONE cell below based on the nvidia-smi output:

In [ ]:
# If CUDA Version showed 13.x: plain install, force torch to match.
%pip install -q --force-reinstall vllm
# Then: Runtime -> Restart session (files survive), rerun from cell 3.

In [ ]:
# If CUDA Version showed 12.x: install the cu128-matched build instead.
# %pip install -q uv
# !uv pip install --system -q vllm --torch-backend=cu128
# Fallback if uv cannot resolve: %pip install -q 'vllm==0.10.2'

In [ ]:
# 3. Code + data (same seed -> byte-identical test set as training time)
!git clone https://github.com/zyziyun/qlora-lab.git
%cd /content/qlora-lab
!python scripts/make_data.py --n 800
import sys; sys.path.insert(0, 'src')

### 4. The adapter is already here
The 1.7B adapter ships with the repo (`outputs/adapter-1.7b`, 67MB), so the
clone in step 3 brought it along - nothing to upload. The 8B adapter exceeds
GitHub's 100MB file limit and is not bundled; only if you want to A/B it too,
drag your `adapters.zip` into the Files pane and run the optional cell below.

In [ ]:
!ls outputs/adapter-1.7b/
# Optional, 8B adapter from a local adapters.zip:
# !unzip -q -o /content/adapters.zip -d /content/qlora-lab

In [ ]:
# 5. Start the vLLM server in the background.
#    --dtype half is REQUIRED on T4 (no bf16 on Turing); harmless on L4.
#    NOTE: Restart session kills this process - rerun this cell after any restart.
import subprocess, time, requests

server = subprocess.Popen([
    'vllm', 'serve', 'Qwen/Qwen3-1.7B',
    '--dtype', 'half',
    '--enable-lora',
    '--lora-modules', 'ticket=outputs/adapter-1.7b',
    '--max-lora-rank', '16',
    '--max-model-len', '4096',
    '--gpu-memory-utilization', '0.85',
    '--port', '8000',
], stdout=open('vllm.log', 'w'), stderr=subprocess.STDOUT)

print('first boot downloads ~3.4GB weights: expect 5-8 min on T4, less after caching')
for i in range(150):
    try:
        if requests.get('http://localhost:8000/v1/models', timeout=2).ok:
            print('server ready'); break
    except Exception:
        pass
    time.sleep(5)
else:
    print('not ready - debug with: !tail -50 vllm.log')

In [ ]:
# 6. The hot-swap proof: one server, two model ids (base + adapter).
!curl -s localhost:8000/v1/models | python3 -m json.tool | grep '"id"'

In [ ]:
# 7. A/B on the same server - the only difference is the model name string.
from qlora_lab import serve, predict, dataset as ds, evaluate as ev

client = serve.openai_client()
NO_THINK = {'chat_template_kwargs': {'enable_thinking': False}}  # Qwen3: no <think> block

test = ds.read_jsonl('data/test.jsonl')[:30]

base_preds  = [predict.extract(client, 'Qwen/Qwen3-1.7B', e['message'], extra_body=NO_THINK) for e in test]
tuned_preds = [predict.extract(client, 'ticket',          e['message'], extra_body=NO_THINK) for e in test]

rb = ev.evaluate(base_preds,  test, in_price=0.05e-6, out_price=0.20e-6)
rt = ev.evaluate(tuned_preds, test, in_price=0.05e-6, out_price=0.20e-6)
print('BASE :', rb.summary())
print('TUNED:', rt.summary())
print(); print(ev.compare(rb, rt))
print('base failures:', [f['reason'][:30] for f in rb.failures[:5]])

In [ ]:
# 8. One visible example - base vs adapter on the same message.
msg = 'Hey team, order 55012 arrived smashed and I am furious. Please advise.'
for m in ['Qwen/Qwen3-1.7B', 'ticket']:
    p = predict.extract(client, m, msg, extra_body=NO_THINK)
    print(f'{m:>18}: {p.raw[:110]}')

Expected: tuned wins the judgment fields (priority/sentiment) with ~32 output
tokens vs the base's longer output, and latency beats single-request HF
generate thanks to vLLM's continuous batching + PagedAttention. If you serve
the 1.7B in production, pair it with the deterministic guardrail from the
README (extracted order_id must be a substring of the message) and escalate
failures via `agent.route_with_fallback`.